# DevCompass Skill Landscape 실험 v2

이 노트북은 채용공고의 기술 동시출현을 군집화하고, 그 결과를 선점기술 후보의 보조 증거로 변환한다.

v2 핵심 원칙:
- 현재 Job Jaccard는 v1 비교 기준으로 유지한다.
- 같은 기업의 반복 공고가 관계를 지배하지 않도록 `pair_company_count`를 계산한다.
- Company-balanced Jaccard를 기본 표현으로 사용하고 filtered PPMI와 비교한다.
- Silhouette뿐 아니라 여러 seed의 ARI 안정성과 군집 크기를 함께 본다.
- KMeans 배정 결과를 `core_member`, `boundary_member`, `cluster_outlier`로 구분한다.
- 전체 기술에는 진단값만 만들고, Gap 분석 후보에만 evidence label을 붙인다.
- UMAP은 시각화용이며 군집화는 SVD embedding에서 수행한다.

## 1. 설정과 라이브러리

`CANDIDATE_SKILL_CODES`에는 Gap 분석이 먼저 선정한 후보의 `skill_code`만 입력한다. 비워 두면 군집 진단까지만 수행하며 선점기술 증거 판정은 하지 않는다.

In [ ]:
from itertools import combinations
from pathlib import Path
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

DATA_PATH = Path(os.getenv(
    "SKILL_LANDSCAPE_CSV",
    "/Users/hojin/Desktop/data_for_cluster_experiment.csv",
))

# 데이터 충분성 기준
MIN_JOB_COUNT = 5
MIN_COMPANY_COUNT = 2
MIN_PAIR_COMPANY_COUNT = 2

# 표현 방식과 모델 탐색 범위
REPRESENTATIONS_TO_COMPARE = ["job_jaccard", "company_jaccard", "filtered_ppmi"]
SELECTED_REPRESENTATION = "company_jaccard"
SVD_COMPONENT_CANDIDATES = [10, 20, 30]
K_CANDIDATES = range(6, 13)
MODEL_RANDOM_STATES = [11, 21, 42, 84, 101]
RANDOM_STATE = 42
MIN_MODEL_STABILITY = 0.80
MIN_CLUSTER_SIZE = 3
MAX_CLUSTER_FRACTION = 0.35
SELECTED_SVD_COMPONENTS = None
SELECTED_K = None

# 군집원 및 후보 증거 판정 기준
OUTLIER_IQR_MULTIPLIER = 1.5
OUTLIER_ROBUST_Z_THRESHOLD = 2.5
BOUNDARY_MARGIN_RATIO = 0.10
MIN_MEMBERSHIP_STABILITY = 0.80
MIN_EVIDENCE_COMPANIES = 3
MIN_SUPPORTING_PAIR_COMPANIES = 2
MAX_SUPPORTING_PAIR_COMPANY_SHARE = 0.70

# DBSCAN은 강제 배정 문제를 확인하는 보조 실험이다.
RUN_DBSCAN_COMPARISON = True
DBSCAN_EPS_VALUES = [0.25, 0.35, 0.45, 0.55, 0.65]
DBSCAN_MIN_SAMPLES = [3, 4, 5]

CANDIDATE_SKILL_CODES = [
    "DOTNET",
    "AMAZON_S3",
    "ANGULAR",
    "CLOUDFLARE",
    "DART",
    "DJANGO",
    "FASTAPI",
    "FLASK",
    "GEMINI",
    "GITHUB_ACTIONS",
    "GITHUB_COPILOT",
    "JUPYTER",
    "LANGCHAIN",
    "LARAVEL",
    "LUA",
    "MONGODB",
    "NESTJS",
    "NEXT_JS",
    "NGINX",
    "NUMPY",
    "PANDAS",
    "POWER_BI",
    "PYTEST",
    "REACT_NATIVE",
    "SASS",
    "SELENIUM",
    "SPRING",
    "SQLITE",
    "SVELTE",
    "SWIFTUI",
    "TAILWIND_CSS",
    "VAULT",
    "VITE",
    "WEBPACK",
]
EXPORT_RESULTS = False

print(f"data: {DATA_PATH}")

## 2. 데이터 로드와 품질 검증

분석 단위는 `공고 1개 × 기술 1개`다. 같은 공고와 기술의 중복 행은 제거하고, 식별자와 이름의 불일치를 검사한다.

In [ ]:
required_columns = {
    "job_id", "board_id", "company_name", "job_title",
    "skill_id", "skill_code", "skill_name",
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV를 찾을 수 없습니다: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")
if raw[list(required_columns)].isna().any().any():
    raise ValueError("필수 컬럼에 결측치가 있습니다.")

duplicate_count = raw.duplicated(["job_id", "skill_id"]).sum()
data = raw.drop_duplicates(["job_id", "skill_id"]).copy()

skill_identity_count = data.groupby("skill_id")[["skill_code", "skill_name"]].nunique()
if (skill_identity_count > 1).any().any():
    raise ValueError("하나의 skill_id에 여러 code 또는 name이 연결되어 있습니다.")

print({
    "rows": len(data),
    "jobs": data["job_id"].nunique(),
    "skills": data["skill_id"].nunique(),
    "companies": data["board_id"].nunique(),
    "removed_duplicates": int(duplicate_count),
})

## 3. 전체 Job × Skill 행렬과 군집 대상 분리

Pair diagnostics는 희귀 기술을 포함한 전체 기술에서 먼저 계산한다. 군집 학습에만 `MIN_JOB_COUNT`, `MIN_COMPANY_COUNT` 기준을 적용해 부분행렬을 사용한다.

이렇게 하면 데이터가 적어 군집에서 제외된 선점후보도 어떤 기술과 어느 기업들에서 함께 등장했는지 확인할 수 있다.

In [ ]:
skill_stats = (
    data.groupby(["skill_id", "skill_code", "skill_name"], as_index=False)
    .agg(job_count=("job_id", "nunique"), company_count=("board_id", "nunique"))
)
skill_stats["eligible_for_clustering"] = (
    (skill_stats["job_count"] >= MIN_JOB_COUNT)
    & (skill_stats["company_count"] >= MIN_COMPANY_COUNT)
)

all_skill_ids = skill_stats["skill_id"].tolist()
eligible_skill_ids = skill_stats.loc[
    skill_stats["eligible_for_clustering"], "skill_id"
].tolist()

full_job_skill = pd.crosstab(data["job_id"], data["skill_id"]).clip(upper=1)
full_job_skill = full_job_skill.reindex(columns=all_skill_ids, fill_value=0).astype(np.uint8)

# 군집 학습은 적격 기술의 부분행렬만 사용한다. 0으로만 채워진 공고 행은 계산에 영향이 없다.
job_skill = full_job_skill.reindex(columns=eligible_skill_ids, fill_value=0)

print(f"전체 pair 진단 기술: {full_job_skill.shape[1]} skills")
print(f"군집 대상 기술: {job_skill.shape[1]} / {full_job_skill.shape[1]}")
print(f"전체 행렬 크기: {full_job_skill.shape[0]} jobs x {full_job_skill.shape[1]} skills")
display(skill_stats.sort_values(["job_count", "company_count"]).head(15))

## 4. 전체 기술 관계 계산과 기업 편향 보정

필터 전 전체 기술에 대해 다음을 계산한 후, 군집 학습 시 적격 기술의 부분행렬만 꺼내 쓴다.

- `job_pair_count`: 같은 공고에 두 기술이 등장한 횟수
- `pair_company_count`: 그 관계가 관측된 서로 다른 기업 수
- `job_jaccard`: 기존 v1 baseline
- `company_jaccard`: 기업별 반복을 1회로 제한한 v2 기본값
- `filtered_ppmi`: 최소 기업 수를 통과한 관계의 PPMI

한 기업의 서로 다른 공고에 기술 A와 B가 따로 등장한 경우는 동시출현으로 세지 않는다.

In [ ]:
def build_jaccard(cooccurrence):
    support = np.diag(cooccurrence).astype(float)
    union = support[:, None] + support[None, :] - cooccurrence
    result = np.divide(
        cooccurrence,
        union,
        out=np.zeros_like(cooccurrence, dtype=float),
        where=union > 0,
    )
    np.fill_diagonal(result, 0.0)
    return result


def build_ppmi(cooccurrence):
    counts = cooccurrence.astype(float).copy()
    np.fill_diagonal(counts, 0.0)
    total = counts.sum()
    if total == 0:
        return np.zeros_like(counts)
    marginals = counts.sum(axis=1)
    expected = np.outer(marginals, marginals) / total
    ratio = np.divide(counts, expected, out=np.ones_like(counts), where=expected > 0)
    ppmi = np.log(np.maximum(ratio, 1.0))
    ppmi[counts == 0] = 0.0
    np.fill_diagonal(ppmi, 0.0)
    return ppmi


all_skill_lookup = skill_stats.set_index("skill_id")
all_skill_ids = np.array(full_job_skill.columns)
full_x = full_job_skill.to_numpy(dtype=np.int64)
full_job_cooccurrence = (full_x.T @ full_x).astype(np.float64)

job_company = (
    data[["job_id", "board_id", "company_name"]]
    .drop_duplicates("job_id")
    .set_index("job_id")
    .reindex(full_job_skill.index)
)
if job_company["board_id"].isna().any():
    raise ValueError("일부 job_id의 기업 정보를 찾을 수 없습니다.")

company_ids = job_company["board_id"].drop_duplicates().tolist()
company_name_by_id = (
    job_company.reset_index()
    .drop_duplicates("board_id")
    .set_index("board_id")["company_name"]
    .to_dict()
)

full_company_job_pair_counts = []
for board_id in company_ids:
    company_mask = job_company["board_id"].to_numpy() == board_id
    company_x = full_x[company_mask]
    full_company_job_pair_counts.append(
        (company_x.T @ company_x).astype(np.float64)
    )
full_company_job_pair_counts = np.stack(full_company_job_pair_counts)
full_pair_company_count = (
    full_company_job_pair_counts > 0
).sum(axis=0).astype(np.float64)

full_job_jaccard = build_jaccard(full_job_cooccurrence)
full_company_jaccard = build_jaccard(full_pair_company_count)
full_ppmi = build_ppmi(full_job_cooccurrence)

full_market_relation_mask = full_pair_company_count >= MIN_PAIR_COMPANY_COUNT
np.fill_diagonal(full_market_relation_mask, False)
full_company_jaccard = np.where(
    full_market_relation_mask, full_company_jaccard, 0.0
)
full_filtered_ppmi = np.where(full_market_relation_mask, full_ppmi, 0.0)

full_feature_matrices = {
    "job_jaccard": full_job_jaccard,
    "company_jaccard": full_company_jaccard,
    "filtered_ppmi": full_filtered_ppmi,
}
for name, matrix in full_feature_matrices.items():
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} 행렬에 NaN 또는 무한대가 있습니다.")

# 전체 167개 기술 관계 진단 파일을 만든다.
pair_rows = []
for i, j in zip(*np.triu_indices(len(all_skill_ids), k=1)):
    job_pair_count = int(full_job_cooccurrence[i, j])
    if job_pair_count == 0:
        continue
    company_counts = full_company_job_pair_counts[:, i, j]
    dominant_index = int(np.argmax(company_counts))
    pair_rows.append({
        "skill_id_a": all_skill_ids[i],
        "skill_name_a": all_skill_lookup.loc[all_skill_ids[i], "skill_name"],
        "skill_id_b": all_skill_ids[j],
        "skill_name_b": all_skill_lookup.loc[all_skill_ids[j], "skill_name"],
        "job_pair_count": job_pair_count,
        "pair_company_count": int(full_pair_company_count[i, j]),
        "top_company": company_name_by_id[company_ids[dominant_index]],
        "top_company_pair_count": int(company_counts[dominant_index]),
        "top_company_share": float(company_counts[dominant_index] / job_pair_count),
        "job_jaccard": float(full_job_jaccard[i, j]),
        "company_jaccard": float(full_company_jaccard[i, j]),
        "filtered_ppmi": float(full_filtered_ppmi[i, j]),
        "passes_company_filter": bool(full_market_relation_mask[i, j]),
    })
pair_diagnostics = pd.DataFrame(pair_rows)

# 아래부터는 군집 적격 기술의 부분행렬이다.
all_position_by_skill_id = {
    skill_id: position for position, skill_id in enumerate(all_skill_ids)
}
eligible_positions = np.array([
    all_position_by_skill_id[skill_id] for skill_id in eligible_skill_ids
])
skill_ids = np.array(eligible_skill_ids)
skill_lookup = all_skill_lookup

feature_matrices = {
    name: matrix[np.ix_(eligible_positions, eligible_positions)]
    for name, matrix in full_feature_matrices.items()
}
job_cooccurrence = full_job_cooccurrence[
    np.ix_(eligible_positions, eligible_positions)
]
pair_company_count = full_pair_company_count[
    np.ix_(eligible_positions, eligible_positions)
]
company_job_pair_counts = full_company_job_pair_counts[
    :, eligible_positions, :
][:, :, eligible_positions]

relation_summary = pd.DataFrame([
    {
        "scope": "all_skills",
        "representation": name,
        "nonzero_relations": int(np.count_nonzero(np.triu(matrix, 1))),
    }
    for name, matrix in full_feature_matrices.items()
] + [
    {
        "scope": "clustering_skills",
        "representation": name,
        "nonzero_relations": int(np.count_nonzero(np.triu(matrix, 1))),
    }
    for name, matrix in feature_matrices.items()
])
display(relation_summary)
display(
    pair_diagnostics.query("pair_company_count < @MIN_PAIR_COMPANY_COUNT")
    .sort_values("job_pair_count", ascending=False)
    .head(15)
)

## 5. 표현 방식, SVD 차원, K의 공동 비교

각 표현 방식에 대해 SVD 10/20/30차원과 K=6~12를 비교한다. 각 조합은 여러 random seed로 반복하고, seed 사이의 Adjusted Rand Index(ARI)를 안정성으로 사용한다.

자동 선택은 다음 조건을 먼저 통과한 조합 안에서만 Silhouette를 비교한다.
- 평균 ARI가 기준 이상
- 지나치게 작은 군집이 없음
- 하나의 군집이 전체 기술의 과도한 비율을 차지하지 않음

최종 판단에서는 아래 표와 군집 해석을 함께 확인한다.

In [ ]:
n_skills = len(skill_ids)
if n_skills < 4:
    raise ValueError("군집화를 수행하기에 기술 수가 너무 적습니다.")

embedding_by_config = {}
label_runs_by_config = {}
model_rows = []

for representation in REPRESENTATIONS_TO_COMPARE:
    matrix = feature_matrices[representation]
    for requested_components in SVD_COMPONENT_CANDIDATES:
        n_components = min(requested_components, n_skills - 1)
        svd_model = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
        embedding_candidate = normalize(svd_model.fit_transform(matrix), norm="l2")
        embedding_by_config[(representation, n_components)] = (svd_model, embedding_candidate)

        for k in [value for value in K_CANDIDATES if 2 <= value < n_skills]:
            labels_per_seed = []
            silhouettes = []
            min_sizes = []
            max_fractions = []
            inertias = []

            for seed in MODEL_RANDOM_STATES:
                model = KMeans(n_clusters=k, n_init=30, random_state=seed)
                labels = model.fit_predict(embedding_candidate)
                sizes = pd.Series(labels).value_counts()
                labels_per_seed.append(labels)
                silhouettes.append(silhouette_score(embedding_candidate, labels))
                min_sizes.append(int(sizes.min()))
                max_fractions.append(float(sizes.max() / n_skills))
                inertias.append(float(model.inertia_))

            ari_values = [
                adjusted_rand_score(labels_a, labels_b)
                for labels_a, labels_b in combinations(labels_per_seed, 2)
            ]
            config_key = (representation, n_components, k)
            label_runs_by_config[config_key] = labels_per_seed
            model_rows.append({
                "representation": representation,
                "svd_components": n_components,
                "k": k,
                "mean_silhouette": float(np.mean(silhouettes)),
                "std_silhouette": float(np.std(silhouettes)),
                "mean_seed_ari": float(np.mean(ari_values)),
                "min_cluster_size": int(min(min_sizes)),
                "max_cluster_fraction": float(max(max_fractions)),
                "mean_inertia": float(np.mean(inertias)),
                "explained_variance": float(svd_model.explained_variance_ratio_.sum()),
            })

model_results = pd.DataFrame(model_rows)
model_results["passes_selection_guardrails"] = (
    (model_results["mean_seed_ari"] >= MIN_MODEL_STABILITY)
    & (model_results["min_cluster_size"] >= MIN_CLUSTER_SIZE)
    & (model_results["max_cluster_fraction"] <= MAX_CLUSTER_FRACTION)
)

selected_results = model_results.query("representation == @SELECTED_REPRESENTATION").copy()
guarded_results = selected_results[selected_results["passes_selection_guardrails"]]
selection_pool = guarded_results if not guarded_results.empty else selected_results

if SELECTED_SVD_COMPONENTS is None or SELECTED_K is None:
    automatic_choice = selection_pool.sort_values(
        ["mean_silhouette", "mean_seed_ari"], ascending=False
    ).iloc[0]
selected_components = (
    SELECTED_SVD_COMPONENTS
    if SELECTED_SVD_COMPONENTS is not None
    else int(automatic_choice["svd_components"])
)
selected_k = SELECTED_K if SELECTED_K is not None else int(automatic_choice["k"])

svd, embedding = embedding_by_config[(SELECTED_REPRESENTATION, selected_components)]
kmeans = KMeans(n_clusters=selected_k, n_init=50, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(embedding)
selected_label_runs = label_runs_by_config[
    (SELECTED_REPRESENTATION, selected_components, selected_k)
]

display(
    selected_results.sort_values(
        ["passes_selection_guardrails", "mean_silhouette", "mean_seed_ari"],
        ascending=[False, False, False],
    ).head(20)
)
print({
    "representation": SELECTED_REPRESENTATION,
    "svd_components": selected_components,
    "k": selected_k,
    "explained_variance": round(float(svd.explained_variance_ratio_.sum()), 3),
})

## 6. DBSCAN 보조 비교

DBSCAN은 KMeans가 모든 기술을 강제로 배정하는 문제를 확인하기 위한 진단이다. 높은 Silhouette와 함께 noise 비율이 지나치게 높다면 주 모델로 채택하지 않는다.

In [ ]:
dbscan_results = pd.DataFrame()
if RUN_DBSCAN_COMPARISON:
    dbscan_rows = []
    for eps in DBSCAN_EPS_VALUES:
        for min_samples in DBSCAN_MIN_SAMPLES:
            labels = DBSCAN(
                eps=eps, min_samples=min_samples, metric="euclidean"
            ).fit_predict(embedding)
            clustered_mask = labels != -1
            n_clusters = len(set(labels[clustered_mask]))
            score = np.nan
            if n_clusters >= 2 and clustered_mask.sum() > n_clusters:
                score = silhouette_score(
                    embedding[clustered_mask], labels[clustered_mask]
                )
            dbscan_rows.append({
                "eps": eps,
                "min_samples": min_samples,
                "n_clusters": n_clusters,
                "noise_ratio": float((labels == -1).mean()),
                "silhouette_without_noise": score,
            })
    dbscan_results = pd.DataFrame(dbscan_rows)
    display(
        dbscan_results.sort_values(
            ["silhouette_without_noise", "noise_ratio"],
            ascending=[False, True],
            na_position="last",
        ).head(10)
    )

## 7. IQR/MAD 기반 군집원 진단과 시각화

거리 상위 25%를 자동으로 outlier 처리하지 않는다. 각 군집의 거리 분포에서 다음 중 하나를 충족할 때만 `cluster_outlier`로 본다.

- `distance > Q3 + 1.5 × IQR`
- `robust_z > 2.5` (`MAD` 기반)

나머지는 centroid margin과 seed 안정성으로 `core_member`와 `boundary_member`를 구분한다. Global neighbor와 같은 군집 내부 neighbor는 별도 컬럼으로 계산한다.

In [ ]:
centroid_distances = kmeans.transform(embedding)
nearest_distances = centroid_distances[np.arange(n_skills), cluster_labels]
sorted_distances = np.sort(centroid_distances, axis=1)
centroid_margin = sorted_distances[:, 1] - sorted_distances[:, 0]
centroid_margin_ratio = np.divide(
    centroid_margin,
    sorted_distances[:, 1],
    out=np.zeros_like(centroid_margin),
    where=sorted_distances[:, 1] > 0,
)

membership_stability = []
for i in range(n_skills):
    baseline_peers = cluster_labels == cluster_labels[i]
    baseline_peers[i] = False
    if not baseline_peers.any():
        membership_stability.append(0.0)
        continue
    run_scores = [
        float(np.mean(labels[baseline_peers] == labels[i]))
        for labels in selected_label_runs
    ]
    membership_stability.append(float(np.mean(run_scores)))

membership = pd.DataFrame({
    "skill_id": skill_ids,
    "cluster_id": cluster_labels,
    "distance_to_centroid": nearest_distances,
    "centroid_margin_ratio": centroid_margin_ratio,
    "membership_stability": membership_stability,
}).join(
    skill_lookup[["skill_code", "skill_name", "job_count", "company_count"]],
    on="skill_id",
)

distance_group = membership.groupby("cluster_id")["distance_to_centroid"]
membership["distance_percentile_in_cluster"] = distance_group.rank(pct=True)
membership["distance_q1_in_cluster"] = distance_group.transform(
    lambda values: values.quantile(0.25)
)
membership["distance_q3_in_cluster"] = distance_group.transform(
    lambda values: values.quantile(0.75)
)
membership["distance_median_in_cluster"] = distance_group.transform("median")
membership["distance_mad_in_cluster"] = distance_group.transform(
    lambda values: (values - values.median()).abs().median()
)
membership["distance_iqr_fence"] = (
    membership["distance_q3_in_cluster"]
    + OUTLIER_IQR_MULTIPLIER
    * (
        membership["distance_q3_in_cluster"]
        - membership["distance_q1_in_cluster"]
    )
)
membership["distance_robust_z"] = np.divide(
    membership["distance_to_centroid"]
    - membership["distance_median_in_cluster"],
    1.4826 * membership["distance_mad_in_cluster"],
    out=np.zeros(len(membership), dtype=float),
    where=membership["distance_mad_in_cluster"].to_numpy() > 0,
)

cluster_coherence = []
same_cluster_strongest_neighbors = []
same_cluster_top_similarities = []
same_cluster_top_pair_companies = []
same_cluster_top_company_shares = []
feature_matrix = feature_matrices[SELECTED_REPRESENTATION]

def pair_top_company_share(i, j):
    pair_jobs = job_cooccurrence[i, j]
    if pair_jobs <= 0:
        return np.nan
    return float(company_job_pair_counts[:, i, j].max() / pair_jobs)


for i in range(n_skills):
    same_cluster_mask = cluster_labels == cluster_labels[i]
    same_cluster_mask[i] = False
    values = feature_matrix[i, same_cluster_mask]
    cluster_coherence.append(float(values.mean()) if values.size else 0.0)

    neighbor_order = np.argsort(feature_matrix[i])[::-1]
    same_cluster_order = [
        j
        for j in neighbor_order
        if same_cluster_mask[j] and feature_matrix[i, j] > 0
    ][:3]
    labels = [
        (
            f"{skill_lookup.loc[skill_ids[j], 'skill_name']} "
            f"(score={feature_matrix[i, j]:.3f}, "
            f"companies={int(pair_company_count[i, j])})"
        )
        for j in same_cluster_order
    ]
    same_cluster_strongest_neighbors.append(", ".join(labels))

    if same_cluster_order:
        top_index = same_cluster_order[0]
        same_cluster_top_similarities.append(float(feature_matrix[i, top_index]))
        same_cluster_top_pair_companies.append(
            int(pair_company_count[i, top_index])
        )
        same_cluster_top_company_shares.append(
            pair_top_company_share(i, top_index)
        )
    else:
        same_cluster_top_similarities.append(np.nan)
        same_cluster_top_pair_companies.append(0)
        same_cluster_top_company_shares.append(np.nan)

membership["cluster_coherence"] = cluster_coherence
membership["same_cluster_strongest_neighbors"] = same_cluster_strongest_neighbors
membership["same_cluster_top_similarity"] = same_cluster_top_similarities
membership["same_cluster_top_pair_companies"] = same_cluster_top_pair_companies
membership["same_cluster_top_company_share"] = same_cluster_top_company_shares

def classify_membership(row):
    is_iqr_outlier = row["distance_to_centroid"] > row["distance_iqr_fence"]
    is_mad_outlier = row["distance_robust_z"] > OUTLIER_ROBUST_Z_THRESHOLD
    if is_iqr_outlier or is_mad_outlier:
        return "cluster_outlier"
    if (
        row["centroid_margin_ratio"] < BOUNDARY_MARGIN_RATIO
        or row["membership_stability"] < MIN_MEMBERSHIP_STABILITY
    ):
        return "boundary_member"
    return "core_member"

membership["membership_quality"] = membership.apply(classify_membership, axis=1)

cluster_rows = []
for cluster_id, group in membership.groupby("cluster_id"):
    indices = np.where(cluster_labels == cluster_id)[0]
    company_events = np.array([
        np.triu(counts[np.ix_(indices, indices)], k=1).sum()
        for counts in company_job_pair_counts
    ])
    total_events = company_events.sum()
    dominant_index = int(np.argmax(company_events)) if total_events > 0 else 0
    cluster_rows.append({
        "cluster_id": cluster_id,
        "skill_count": len(group),
        "core_count": int((group["membership_quality"] == "core_member").sum()),
        "boundary_count": int((group["membership_quality"] == "boundary_member").sum()),
        "outlier_count": int((group["membership_quality"] == "cluster_outlier").sum()),
        "mean_coherence": float(group["cluster_coherence"].mean()),
        "dominant_company": company_name_by_id[company_ids[dominant_index]],
        "top_company_share": float(company_events[dominant_index] / total_events)
        if total_events > 0 else np.nan,
    })

    print(f"\nCluster {cluster_id} ({len(group)} skills)")
    for quality in ["core_member", "boundary_member", "cluster_outlier"]:
        names = group.loc[
            group["membership_quality"] == quality, "skill_name"
        ].tolist()
        print(f"  {quality}: {', '.join(names) if names else '-'}")

cluster_summary = pd.DataFrame(cluster_rows).sort_values("cluster_id")
display(cluster_summary)

try:
    import umap
    projector = umap.UMAP(
        n_components=2, metric="cosine", random_state=RANDOM_STATE, n_jobs=1
    )
    projection_name = "UMAP"
except ImportError:
    projector = PCA(n_components=2, random_state=RANDOM_STATE)
    projection_name = "PCA fallback"

coordinates = projector.fit_transform(embedding)
membership[["embedding_x", "embedding_y"]] = coordinates

plt.figure(figsize=(14, 9))
sns.scatterplot(
    data=membership,
    x="embedding_x",
    y="embedding_y",
    hue="cluster_id",
    style="membership_quality",
    size="job_count",
    sizes=(35, 350),
    palette="tab10",
    alpha=0.8,
    legend="brief",
)
for row in membership.nlargest(25, "job_count").itertuples():
    plt.text(row.embedding_x, row.embedding_y, row.skill_name, fontsize=8)
plt.title(
    f"Skill Landscape ({projection_name}, K={selected_k}, {SELECTED_REPRESENTATION})"
)
plt.tight_layout()
plt.show()

## 8. Global neighbor와 같은 군집 neighbor를 분리한 후보 판정

전체 기술에서 가장 가까운 `global neighbor`는 시장의 직접 연결을 설명한다. `same-cluster neighbor`는 현재 군집 소속을 뒷받침하는 관계다.

Supporting 판정의 기업 확산 조건에는 같은 군집 내부 neighbor만 사용한다. 전체 기술에는 진단값만 만들고, Gap 분석 후보에만 evidence label을 붙인다.

- `insufficient_evidence`: 후보의 데이터가 적어 군집 대상에서 제외됨
- `weak_evidence`: 군집 내부 기업 확산, 소속 안정성 또는 기업 편향 조건이 약함
- `supporting_evidence`: 여러 기업에서 같은 군집 기술과 연결되고 군집 소속도 안정적임

In [ ]:
# Global neighbor는 필터 전 전체 기술 행렬에서 계산한다.
global_rows = []
global_matrix = full_feature_matrices[SELECTED_REPRESENTATION]
for i, skill_id in enumerate(all_skill_ids):
    neighbor_order = np.argsort(global_matrix[i])[::-1]
    neighbor_order = [
        j for j in neighbor_order if global_matrix[i, j] > 0
    ][:3]
    labels = [
        (
            f"{all_skill_lookup.loc[all_skill_ids[j], 'skill_name']} "
            f"(score={global_matrix[i, j]:.3f}, "
            f"companies={int(full_pair_company_count[i, j])})"
        )
        for j in neighbor_order
    ]
    row = {
        "skill_id": skill_id,
        "global_strongest_neighbors": ", ".join(labels),
        "global_top_similarity": np.nan,
        "global_top_pair_companies": 0,
        "global_top_company_share": np.nan,
    }
    if neighbor_order:
        top_index = neighbor_order[0]
        pair_jobs = full_job_cooccurrence[i, top_index]
        top_jobs = full_company_job_pair_counts[:, i, top_index].max()
        row.update({
            "global_top_similarity": float(global_matrix[i, top_index]),
            "global_top_pair_companies": int(
                full_pair_company_count[i, top_index]
            ),
            "global_top_company_share": float(top_jobs / pair_jobs)
            if pair_jobs > 0 else np.nan,
        })
    global_rows.append(row)
global_neighbor_diagnostics = pd.DataFrame(global_rows)

skill_diagnostics = (
    skill_stats
    .merge(global_neighbor_diagnostics, on="skill_id", how="left")
    .merge(
        membership[[
            "skill_id",
            "cluster_id",
            "distance_to_centroid",
            "distance_percentile_in_cluster",
            "distance_iqr_fence",
            "distance_robust_z",
            "centroid_margin_ratio",
            "membership_stability",
            "membership_quality",
            "cluster_coherence",
            "same_cluster_strongest_neighbors",
            "same_cluster_top_similarity",
            "same_cluster_top_pair_companies",
            "same_cluster_top_company_share",
            "embedding_x",
            "embedding_y",
        ]],
        on="skill_id",
        how="left",
    )
)

def classify_candidate_evidence(row):
    if pd.isna(row["cluster_id"]):
        return "insufficient_evidence"
    is_supported = (
        row["company_count"] >= MIN_EVIDENCE_COMPANIES
        and row["same_cluster_top_pair_companies"]
        >= MIN_SUPPORTING_PAIR_COMPANIES
        and row["same_cluster_top_company_share"]
        <= MAX_SUPPORTING_PAIR_COMPANY_SHARE
        and row["centroid_margin_ratio"] >= BOUNDARY_MARGIN_RATIO
        and row["membership_stability"] >= MIN_MEMBERSHIP_STABILITY
        and row["membership_quality"] != "cluster_outlier"
    )
    return "supporting_evidence" if is_supported else "weak_evidence"

candidate_evidence = skill_diagnostics[
    skill_diagnostics["skill_code"].isin(CANDIDATE_SKILL_CODES)
].copy()
candidate_evidence["evidence_label"] = candidate_evidence.apply(
    classify_candidate_evidence, axis=1
)

missing_candidates = sorted(
    set(CANDIDATE_SKILL_CODES) - set(candidate_evidence["skill_code"])
)
if missing_candidates:
    print(f"CSV에 없는 후보: {missing_candidates}")

display(
    skill_diagnostics.sort_values(
        ["cluster_id", "membership_quality", "distance_to_centroid"],
        na_position="last",
    )
)

## 9. 후보 확인 및 선택적 저장

`CANDIDATE_SKILL_CODES`가 비어 있으면 evidence label을 만들지 않는다. Gap 분석 후보를 입력한 후에만 후보별 산업 연결 근거를 확인한다. 결과를 검토한 뒤 `EXPORT_RESULTS=True`로 변경하면 진단 테이블을 저장할 수 있다.

In [ ]:
if CANDIDATE_SKILL_CODES:
    display(candidate_evidence[[
        "skill_code",
        "skill_name",
        "job_count",
        "company_count",
        "cluster_id",
        "membership_quality",
        "distance_to_centroid",
        "distance_iqr_fence",
        "distance_robust_z",
        "centroid_margin_ratio",
        "membership_stability",
        "global_strongest_neighbors",
        "global_top_similarity",
        "global_top_pair_companies",
        "global_top_company_share",
        "same_cluster_strongest_neighbors",
        "same_cluster_top_similarity",
        "same_cluster_top_pair_companies",
        "same_cluster_top_company_share",
        "evidence_label",
    ]])
    display(
        candidate_evidence["evidence_label"]
        .value_counts()
        .rename_axis("label")
        .reset_index(name="candidate_count")
    )
else:
    print(
        "CANDIDATE_SKILL_CODES가 비어 있습니다. "
        "전체 기술에는 진단값만 생성했고 evidence label은 부여하지 않았습니다."
    )

if EXPORT_RESULTS:
    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)
    skill_diagnostics.to_csv(
        output_dir / "skill_cluster_diagnostics.csv", index=False
    )
    candidate_evidence.to_csv(
        output_dir / "candidate_cluster_evidence.csv", index=False
    )
    model_results.to_csv(
        output_dir / "skill_cluster_model_selection.csv", index=False
    )
    pair_diagnostics.to_csv(
        output_dir / "skill_pair_diagnostics.csv", index=False
    )
    cluster_summary.to_csv(
        output_dir / "skill_cluster_summary.csv", index=False
    )
    print(f"저장 완료: {output_dir.resolve()}")